In [1]:
!pip install mediapipe opencv-python numpy tqdm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install opencv-python numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip uninstall mediapipe -y
!pip install mediapipe==0.10.14

Found existing installation: mediapipe 0.10.14
Uninstalling mediapipe-0.10.14:
  Successfully uninstalled mediapipe-0.10.14
  Using cached mediapipe-0.10.14-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.7 kB)
Using cached mediapipe-0.10.14-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (35.7 MB)


In [2]:
import mediapipe
print(mediapipe)
print(mediapipe.__file__)

<module 'mediapipe' from 'c:\\Users\\Rishu\\AppData\\Local\\Programs\\Python\\Python310\\lib\\site-packages\\mediapipe\\__init__.py'>
c:\Users\Rishu\AppData\Local\Programs\Python\Python310\lib\site-packages\mediapipe\__init__.py


In [3]:
import mediapipe as mp

print("Mediapipe version:", mp.__version__)
print("Solutions available:", dir(mp))

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True)

print("Hands module loaded successfully")

Mediapipe version: 0.10.14
Solutions available: ['CalculatorGraph', 'GraphInputStreamAddMode', 'Image', 'ImageFormat', 'ImageFrame', 'Matrix', 'Packet', 'Timestamp', 'ValidatedGraphConfig', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'calculators', 'model_ckpt_util', 'packet_creator', 'packet_getter', 'resource_util', 'solutions', 'tasks']
Hands module loaded successfully


In [4]:
import os
import cv2
import json
import mediapipe as mp

mp_hands = mp.solutions.hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.7
)

DATASET_PATH = r"C:\Users\Rishu\OneDrive\Desktop\minor_sem6\Bharatanatyam-Mudra-Dataset"

dataset = []

def extract_landmarks(image):

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = mp_hands.process(image_rgb)

    if result.multi_hand_landmarks:

        hand = result.multi_hand_landmarks[0]

        landmarks = []

        for lm in hand.landmark:
            landmarks.extend([lm.x, lm.y, lm.z])

        return landmarks

    return None

import numpy as np

def normalize_landmarks(landmarks):

    landmarks = np.array(landmarks).reshape(21,3)

    wrist = landmarks[0]

    shifted = landmarks - wrist

    scale = np.max(np.linalg.norm(shifted, axis=1))

    if scale == 0:
        scale = 1

    normalized = shifted / scale

    return normalized.flatten().tolist()

for mudra in os.listdir(DATASET_PATH):

    folder = os.path.join(DATASET_PATH, mudra)

    if not os.path.isdir(folder):
        continue

    print("Processing:", mudra)
    for img in os.listdir(folder):

        img_path = os.path.join(folder, img)

        image = cv2.imread(img_path)

        if image is None:
            continue

        landmarks = extract_landmarks(image)

        if landmarks:
            landmarks = normalize_landmarks(landmarks)
            dataset.append({
                "mudra": mudra,
                "landmarks": landmarks
            })
    
path = r"C:\Users\Rishu\OneDrive\Desktop\minor_sem6\models\mudra_dataset.json"

with open(path, "w") as f:
    json.dump(dataset, f)

print("Dataset created")

Processing: Alapadmam(1)


c:\Users\Rishu\AppData\Local\Programs\Python\Python310\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing: Anjali(1)
Processing: Aralam(1)
Processing: Ardhachandran(1)
Processing: Ardhapathaka(1)
Processing: Berunda(1)
Processing: Bramaram(1)
Processing: Chakra(1)
Processing: Chandrakala(1)
Processing: Chaturam(1)
Processing: Garuda(1)
Processing: Hamsapaksha(1)
Processing: Hamsasyam(1)
Processing: Kangulam(1)
Processing: Kapith(1)
Processing: Kapotham(1)
Processing: Karkatta(1)
Processing: Kartariswastika(1)
Processing: Katakamukha_1
Processing: Katakamukha_2
Processing: Katakamukha_3
Processing: Katakavardhana(1)
Processing: Katrimukha(1)
Processing: Khatva(1)
Processing: Kilaka(1)
Processing: Kurma(1)
Processing: Matsya(1)
Processing: Mayura(1)
Processing: Mrigasirsha(1)
Processing: Mukulam(1)
Processing: Mushti(1)
Processing: Nagabandha(1)
Processing: Padmakosha(1)
Processing: Pasha(1)
Processing: Pathaka(1)
Processing: Pushpaputa(1)
Processing: Sakata(1)
Processing: Samputa(1)
Processing: Sarpasirsha(1)
Processing: Shanka(1)
Processing: Shivalinga(1)
Processing: Shukatundam

In [9]:
import json
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


path = r"C:\Users\Rishu\OneDrive\Desktop\minor_sem6\models\mudra_dataset.json"
with open(path) as f:
    data = json.load(f)

X = []
y = []

for item in data:

    X.append(item["landmarks"])
    y.append(item["mudra"])


X = np.array(X)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Splitting done. Training model...")

model = RandomForestClassifier(n_estimators=500, max_depth=20, random_state=42)

model.fit(X_train, y_train)
model_path = r"C:\Users\Rishu\OneDrive\Desktop\minor_sem6\models\mudra_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump((model,label_encoder),f)

print("Model trained")

Splitting done. Training model...
Model trained


In [10]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy * 100, "%")

Accuracy: 96.33838383838383 %


In [11]:
import cv2
import mediapipe as mp
import numpy as np
import pickle

# ===============================
# Load trained model
# ===============================
model_path = r"C:\Users\Rishu\OneDrive\Desktop\minor_sem6\models\mudra_model.pkl"
with open(model_path, "rb") as f:
    model, label_encoder = pickle.load(f)

print("Model loaded successfully")

# ===============================
# Initialize MediaPipe Hands
# ===============================

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True)

# ===============================
# Function: Extract Hand Landmarks
# ===============================

def extract_landmarks(image):

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    result = hands.process(image_rgb)

    if result.multi_hand_landmarks:

        hand_landmarks = result.multi_hand_landmarks[0]

        landmarks = []

        for lm in hand_landmarks.landmark:
            landmarks.extend([lm.x, lm.y, lm.z])

        return landmarks

    return None


# ===============================
# Function: Normalize Landmarks
# ===============================

def normalize_landmarks(landmarks):

    landmarks = np.array(landmarks).reshape(21, 3)

    wrist = landmarks[0]

    shifted = landmarks - wrist

    scale = np.max(np.linalg.norm(shifted, axis=1))

    if scale == 0:
        scale = 1

    normalized = shifted / scale

    return normalized.flatten()


def predict_mudra(image_path):

    image = cv2.imread(image_path)

    if image is None:
        print("Error: Image not found")
        return

    landmarks = extract_landmarks(image)

    if landmarks is None:
        print("No hand detected")
        return

    features = normalize_landmarks(landmarks)
    
    prediction = model.predict([features])

    mudra_name = label_encoder.inverse_transform(prediction)[0]

    print("Predicted Mudra:", mudra_name)

    return mudra_name


if __name__ == "__main__":

    image_path = r"C:\Users\Rishu\OneDrive\Desktop\minor_sem6\Bharatanatyam-Mudra-Dataset\Alapadmam(1)\Alapadmam_1.jpg"

    mudra = predict_mudra(image_path)

    if mudra:
        print("Detected Mudra:", mudra)

Model loaded successfully
Predicted Mudra: Alapadmam(1)
Detected Mudra: Alapadmam(1)


c:\Users\Rishu\AppData\Local\Programs\Python\Python310\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
